In [1]:
#load the dataset
#make the dataset tidy or demonstrate that it was already tidy
#demonstrate the size of the dataset
#find out how much data is missing, where its missing, and if its missing at random or seems to have any systematic relationships in its missingness
#find and flag any outliers or suspicious entries
#clean the data or demonstrate that it was already clean. You may choose how to deal with missingness (dropna of fillna... how='any' or 'all') and you should justify your choice in some way
#You will load raw data from data/00-raw/, you will (optionally) write intermediate stages of your work to data/01-interim and you will write the final fully wrangled version of your data to data/02-processed


In [2]:
import pandas as pd
import numpy as np

In [3]:
#load the dataset
df_w = pd.read_csv("data/00-raw/NYC2023Weather.csv")

In [4]:
#size
len(df_w)

33284

In [5]:
#description of data missing per row
df_w.isna().sum(axis=1).describe().round(1)

count    33284.0
mean        14.6
std          1.6
min          7.0
25%         14.0
50%         15.0
75%         16.0
max         16.0
dtype: float64

In [6]:
#data missing per column
df_w.isna().sum(axis=0)

STATION          0
NAME             0
LATITUDE         0
LONGITUDE        0
ELEVATION        0
DATE             0
DAPR         32700
MDPR         32704
PRCP           779
SNOW         11522
SNWD         24336
TAVG         32189
TMAX         27747
TMIN         27745
TOBS         30817
WT01         32536
WT02         33225
WT03         33044
WT04         33256
WT05         33282
WT06         33284
WT08         33073
WT11         33273
dtype: int64

In [7]:
#most of missing data is in variables we do not need
#how many stations have all the data we need?
df_w_na = df_w.dropna(subset = ['STATION','NAME','LONGITUDE','LATITUDE','DATE','TMAX','TMIN', 'PRCP','SNOW'])
len(df_w_na['NAME'].unique())

11

In [8]:
## make the dataset tidy
#trim unnecessary data and rename columns
rename_map = {'STATION': 'Station', 'NAME': 'Name', 'LATITUDE': 'Latitude', 'LONGITUDE': 'Longitude', 'DATE': 'Date', 'PRCP': 'Precipitation', 'SNOW': 'Snow', 'TAVG': 'Avg_Temp', 'TMAX': 'Max_Temp', 'TMIN': 'Min_Temp'}
col_keep = ['STATION', 'NAME', 'LATITUDE', 'LONGITUDE', 'DATE', 'PRCP', 'SNOW', 'TAVG', 'TMAX', 'TMIN']

df_w = df_w[col_keep].rename(columns = rename_map)
df_w.head()

df_w = df_w.dropna(subset = ['Station','Name','Latitude','Longitude','Date','Max_Temp','Min_Temp', 'Precipitation','Snow'])

#categorize snow and rainy days in new column (boolean)
df_w['Rain?'] = np.where(df_w['Precipitation'] > 0, 'yes', 'no')
df_w['Snow?'] = np.where(df_w['Snow'] > 0, 'yes', 'no')

#indexing fix
df_w = df_w.reset_index(drop = True)

#some days are missing average values, let's assume the mean of Max_Temp and Min_Temp is Avg_Temp for those days only
temp_means = (df_w['Max_Temp'] + df_w['Min_Temp']) / 2
df_w['Avg_Temp'] = df_w['Avg_Temp'].fillna(temp_means)

#make sure all dates are in proper format and convert to date time
df_w['Date'] = pd.to_datetime(df_w['Date'])

# split dataset by state
df_ny_w = df_w[df_w['Name'].str.contains('NY')]
df_nj_w = df_w[df_w['Name'].str.contains('NJ')]

#narrow down to manhattan, using Central Park station data
df_central_park = df_ny_w[df_ny_w['Name'] == 'NY CITY CENTRAL PARK, NY US']
#another indexing fix
df_central_park = df_central_park.reset_index(drop = True)

#preview
df_central_park.head()

,Station,Name,Latitude,Longitude,Date,Precipitation,Snow,Avg_Temp,Max_Temp,Min_Temp,Rain?,Snow?
0,USW00094728,"NY CITY CENTRAL PARK, NY US",40.77898,-73.96925,2023-01-01,0.00,0.0,52.0,55.0,49.0,no,no
1,USW00094728,"NY CITY CENTRAL PARK, NY US",40.77898,-73.96925,2023-01-02,0.02,0.0,52.5,56.0,49.0,yes,no
2,USW00094728,"NY CITY CENTRAL PARK, NY US",40.77898,-73.96925,2023-01-03,0.42,0.0,52.5,58.0,47.0,yes,no
3,USW00094728,"NY CITY CENTRAL PARK, NY US",40.77898,-73.96925,2023-01-04,0.02,0.0,57.5,66.0,49.0,yes,no
4,USW00094728,"NY CITY CENTRAL PARK, NY US",40.77898,-73.96925,2023-01-05,0.01,0.0,47.0,50.0,44.0,yes,no


In [9]:
#check for inconsistencies
df_central_park[['Precipitation','Snow', 'Avg_Temp', 'Max_Temp', 'Min_Temp']].describe()

,Precipitation,Snow,Avg_Temp,Max_Temp,Min_Temp
count,365.000000,365.000000,365.000000,365.000000,365.000000
mean,0.162438,0.006301,57.997260,64.926027,51.068493
std,0.458121,0.069841,14.480709,15.341444,14.045382
min,0.000000,0.000000,15.000000,27.000000,3.000000
25%,0.000000,0.000000,46.000000,53.000000,39.000000
50%,0.000000,0.000000,57.000000,64.000000,50.000000
75%,0.070000,0.000000,71.000000,78.000000,64.000000
max,5.480000,0.900000,85.000000,93.000000,77.000000


In [10]:
#according to google, 2/4/23-2/5/23 had extreme low temperatures
df_central_park.loc[df_central_park['Min_Temp'] == df_central_park['Min_Temp'].min()]

#may be a potential future outlier but is not suspicious

,Station,Name,Latitude,Longitude,Date,Precipitation,Snow,Avg_Temp,Max_Temp,Min_Temp,Rain?,Snow?
34,USW00094728,"NY CITY CENTRAL PARK, NY US",40.77898,-73.96925,2023-02-04,0.0,0.0,15.0,27.0,3.0,no,no


In [ ]:
#save to data/02-processed (unhash to save)
#park.to_csv('data/02-processed/CLEANED_CentralPark_Weather2023.csv', index = False)